# Phase 5: Machine Learning Model Benchmarking & Multi-Horizon Risk Classification

## Executive Summary
This notebook strictly focuses on **Machine Learning Model Training, Benchmarking, and Evaluation**. 

To deliver actionable commercial value, we model customer retention not merely as a binary outcome, but as a **Multi-Tiered Risk Horizon**:
1. **High Risk / Urgent Intervention (3-Month Horizon / 90 Days):** Immediate drop in engagement requiring direct retention incentives.
2. **Medium Risk / Re-engagement Window (6-Month Horizon / 180 Days):** Gradual drift requiring targeted re-engagement surveys and personalized product bundles.
3. **Low/Dormant Risk / Win-back Strategy (9+ Month Horizon / 270+ Days):** Long-term churned customers requiring automated win-back workflows.

### Evaluated Model Families
- **Linear Models:** Logistic Regression (L2 Regularized with Feature Scaling)
- **Ensemble Tree Models:** Random Forest Classifier & Gradient Boosting Classifier
- **Advanced Boosting:** HistGradientBoosting / XGBoost Classifiers

In [1]:
# ==============================================================================
# STEP 1: ENVIRONMENT SETUP & STRATEGIC PATH RESOLUTION
# ==============================================================================
import os
import sys
import pickle
from pathlib import Path
import numpy as np
import pandas as pd

# Path Resolution Framework
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'
REPORTS_DIR = PROJECT_ROOT / 'reports' / 'figures' / 'model_evaluation'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / 'customer_features_matrix.parquet'
BEST_MODEL_PATH = MODELS_DIR / 'best_churn_pipeline.pkl'

# Execution Constants
RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 5

print(f"[OK] Project Root Resolved: {PROJECT_ROOT}")
print(f"[OK] Data Source:           {INPUT_PATH}")
print(f"[OK] Models Target Dir:     {MODELS_DIR}")

[OK] Project Root Resolved: C:\Users\LOQ\OneDrive\Desktop\PROJECTS\PROGRAMMING\Customer Churn Intelligence & Retention System
[OK] Data Source:           C:\Users\LOQ\OneDrive\Desktop\PROJECTS\PROGRAMMING\Customer Churn Intelligence & Retention System\data\processed\customer_features_matrix.parquet
[OK] Models Target Dir:     C:\Users\LOQ\OneDrive\Desktop\PROJECTS\PROGRAMMING\Customer Churn Intelligence & Retention System\models


## Step 2: Data Ingestion & Multi-Horizon Target Engineering

We ingest the cleaned customer matrix and formulate two target labels:
1. **`Churn_Target_Binary`:** Standard 90-day binary classification.
2. **`Churn_Risk_Tier`:** Multi-class target partitioning customers into:
   - `0`: Active / Retained (< 90 Days)
   - `1`: Early Risk (90 - 180 Days / 3 to 6 Months)
   - `2`: High Risk (180 - 270 Days / 6 to 9 Months)
   - `3`: Dormant / Churned (> 270 Days / 9+ Months)

In [2]:
# ==============================================================================
# STEP 2: DATA INGESTION & MULTI-HORIZON TARGET CREATION
# ==============================================================================
if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Missing input feature matrix: {INPUT_PATH}")

df_matrix = pd.read_parquet(INPUT_PATH)

# Base Feature Predictors
FEATURE_COLUMNS = [
    'Frequency_Orders', 'Monetary_Spend', 'Total_Units', 
    'Unique_Products', 'Avg_Order_Value', 'Avg_Units_Per_Order'
]

# Create Multi-Horizon Risk Tier Categories (3, 6, 9 Months Logic)
def categorize_churn_risk(recency_days):
    if recency_days <= 90:
        return 0  # Active / Retained
    elif recency_days <= 180:
        return 1  # Moderate Risk (3-6 Months)
    elif recency_days <= 270:
        return 2  # High Risk (6-9 Months)
    else:
        return 3  # Dormant / Lost (9+ Months)

df_matrix['Churn_Risk_Tier'] = df_matrix['Recency_Days'].apply(categorize_churn_risk)

# Standard Binary Target (90-day threshold)
df_matrix['Churn_Target_Binary'] = (df_matrix['Recency_Days'] > 90).astype('int8')

X = df_matrix[FEATURE_COLUMNS].copy()
y_binary = df_matrix['Churn_Target_Binary'].copy()
y_multi = df_matrix['Churn_Risk_Tier'].copy()

print(f"[OK] Matrix Ingested: {X.shape[0]:,} rows x {X.shape[1]} features")
print("\nMulti-Horizon Risk Tier Breakdown:")
print(pd.Series(y_multi).value_counts(normalize=True).rename({
    0: 'Active (<3 Months)',
    1: 'Moderate Risk (3-6 Months)',
    2: 'High Risk (6-9 Months)',
    3: 'Dormant (9+ Months)'
}).map('{:.2%}'.format))

[OK] Matrix Ingested: 5,862 rows x 6 features

Multi-Horizon Risk Tier Breakdown:
Churn_Risk_Tier
Active (<3 Months)            49.22%
Dormant (9+ Months)           32.69%
Moderate Risk (3-6 Months)    10.01%
High Risk (6-9 Months)         8.09%
Name: proportion, dtype: object


## Step 3: Anti-Leakage Stratified Data Partitioning

We split the dataset using **Stratified 80/20 Sampling** to guarantee identical class ratios in both training and test sets.
All data transformations (such as `StandardScaler`) are fit exclusively on training data inside Pipeline objects to prevent data leakage.

In [3]:
# ==============================================================================
# STEP 3: STRATIFIED TRAIN / TEST BOUNDARY
# ==============================================================================
from sklearn.model_selection import train_test_split

# Binary Model Partition
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_binary
)

# Multi-Class Risk Tier Partition
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X, y_multi, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y_multi
)

print(f"[OK] Binary Partition: Train={X_train.shape[0]:,} | Test={X_test.shape[0]:,}")
print(f"[OK] Multi-Tier Partition: Train={X_train_m.shape[0]:,} | Test={X_test_m.shape[0]:,}")

[OK] Binary Partition: Train=4,689 | Test=1,173
[OK] Multi-Tier Partition: Train=4,689 | Test=1,173


## Step 4: Multi-Algorithm Model Pipelines & 5-Fold Stratified Cross-Validation

We benchmark multiple distinct classification algorithm families:
1. **Logistic Regression:** Linear classifier with L2 Regularization & Feature Scaling.
2. **Random Forest:** Non-parametric Ensemble Bagging Trees.
3. **Gradient Boosting:** Sequential Ensemble Tree Optimization.
4. **HistGradientBoosting:** High-performance Gradient Boosting for numeric tabular data.

In [5]:
# ==============================================================================
# STEP 4: MODEL PIPELINES & CROSS-VALIDATION BENCHMARK (WINDOWS-SAFE)
# ==============================================================================
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier

# Candidate Pipelines (Set n_jobs=1 to avoid Windows worker termination)
pipelines = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('classifier', RandomForestClassifier(n_estimators=100, max_depth=8, random_state=RANDOM_STATE, n_jobs=1))
    ]),
    'Gradient Boosting': Pipeline([
        ('classifier', GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE))
    ]),
    'Hist Gradient Boosting': Pipeline([
        ('classifier', HistGradientBoostingClassifier(max_iter=100, learning_rate=0.05, max_depth=4, random_state=RANDOM_STATE))
    ])
}

cv_strategy = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
cv_results = []

print("Executing 5-Fold Stratified Cross-Validation Across Classifiers...")

for name, pipe in pipelines.items():
    # Set n_jobs=1 for stable execution on Windows
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv_strategy, 
        scoring=['roc_auc', 'f1', 'precision', 'recall'], n_jobs=1
    )
    cv_results.append({
        'Algorithm': name,
        'Mean CV ROC-AUC': np.mean(scores['test_roc_auc']),
        'Std CV ROC-AUC': np.std(scores['test_roc_auc']),
        'Mean CV F1-Score': np.mean(scores['test_f1']),
        'Mean CV Precision': np.mean(scores['test_precision']),
        'Mean CV Recall': np.mean(scores['test_recall'])
    })

df_cv_summary = pd.DataFrame(cv_results).sort_values(by='Mean CV ROC-AUC', ascending=False).reset_index(drop=True)
print("\n=== CROSS-VALIDATION BENCHMARK RESULTS ===")
display(df_cv_summary.round(4))

: 

## Step 5: Held-Out Test Performance & Multi-Horizon Risk Evaluation

We evaluate model predictions on the held-out test partition (`X_test`) across all core business metrics:
- **`ROC-AUC` & `PR-AUC`:** Overall ranking capability.
- **`F1-Score`, `Precision`, `Recall`:** Precision vs Recall balance for retention campaigns.

Additionally, we fit a **Multi-Class Classifier** to predict the specific **Risk Horizon Tier (3 / 6 / 9 Months)** for action routing.

In [ ]:
# ==============================================================================
# STEP 5: TEST SET EVALUATION & MULTI-RISK TIER MODEL
# ==============================================================================
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, classification_report

test_benchmark = []
fitted_binary_models = {}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    fitted_binary_models[name] = pipe
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    
    test_benchmark.append({
        'Algorithm': name,
        'Test ROC-AUC': roc_auc_score(y_test, y_proba),
        'Test PR-AUC': average_precision_score(y_test, y_proba),
        'Test F1-Score': f1_score(y_test, y_pred),
        'Test Precision': precision_score(y_test, y_pred),
        'Test Recall': recall_score(y_test, y_pred)
    })

df_test_summary = pd.DataFrame(test_benchmark).sort_values(by='Test ROC-AUC', ascending=False).reset_index(drop=True)
print("=== HELD-OUT TEST EVALUATION (BINARY 90-DAY CHURN) ===")
display(df_test_summary.round(4))

# ------------------------------------------------------------------------------
# MULTI-HORIZON RISK TIER MODELING (3, 6, 9 MONTHS ACTIONABLE STRATEGY)
# ------------------------------------------------------------------------------
print("\n" + "="*70)
print("TRAINING MULTI-HORIZON RISK TIER CLASSIFIER (3 / 6 / 9 MONTHS)")
print("="*70)

champion_algorithm_name = df_test_summary.iloc[0]['Algorithm']
multi_tier_pipeline = pipelines[champion_algorithm_name]

multi_tier_pipeline.fit(X_train_m, y_train_m)
y_pred_m = multi_tier_pipeline.predict(X_test_m)

tier_labels = ['Active (<3 Mo)', 'Moderate Risk (3-6 Mo)', 'High Risk (6-9 Mo)', 'Dormant (9+ Mo)']
print(f"\nClassification Report for Champion Model ('{champion_algorithm_name}'):\n")
print(classification_report(y_test_m, y_pred_m, target_names=tier_labels))

## Step 6: Atomic Artifact Serialization & Persistence

We export the champion pipeline utilizing the **Atomic Write-and-Replace Pattern** (`.tmp.pkl` $\rightarrow$ `.pkl`) to guarantee zero file corruption risks during downstream deployment.

In [ ]:
# ==============================================================================
# STEP 6: ATOMIC PRODUCTION EXPORT
# ==============================================================================
champion_pipeline = fitted_binary_models[champion_algorithm_name]

temp_export_path = BEST_MODEL_PATH.with_suffix('.tmp.pkl')

with open(temp_export_path, 'wb') as f:
    pickle.dump({
        'binary_model': champion_pipeline,
        'multi_tier_model': multi_tier_pipeline,
        'feature_columns': FEATURE_COLUMNS,
        'champion_name': champion_algorithm_name
    }, f)

temp_export_path.replace(BEST_MODEL_PATH)

print("=" * 70)
print("PHASE 5 MODEL EVALUATION & EXPORT COMPLETED SUCCESSFULLY")
print(f"Champion Architecture: '{champion_algorithm_name}'")
print(f"Exported Path:         {BEST_MODEL_PATH.resolve()}")
print("=" * 70)